# ローマ銀貨データ分析

クリオダイナミクスを用いたローマ帝国衰退の数理モデリング

## 目次
1. データの読み込み
2. 探索的データ分析（EDA）
3. 銀含有率の時系列分析
4. 統計検定
5. 可視化

In [ ]:
# ライブラリのインポート
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import sys
from pathlib import Path

# プロジェクトパスを追加
sys.path.insert(0, str(Path.cwd().parent))

from src.config import *
from src.data_loader import RomanCoinDataLoader
from src.visualization import SilverContentVisualizer, create_all_figures

# 日本語フォント設定
plt.rcParams['font.family'] = ['MS Gothic', 'Yu Gothic', 'Meiryo', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False

%matplotlib inline

## 1. データの読み込み

In [ ]:
# データローダーの初期化
loader = RomanCoinDataLoader()

# デナリウス貨データの読み込み
denarii = loader.load_denarii()
print(f"デナリウス貨データ: {len(denarii)} 件")

# 属州貨幣データの読み込み
provincial = loader.load_provincial()
print(f"属州貨幣データ: {len(provincial)} 件")

In [ ]:
# カラム一覧を確認
print("=== デナリウス貨のカラム ===")
print(denarii.columns.tolist())

In [ ]:
# データの前処理
denarii = loader.preprocess(denarii)
provincial = loader.preprocess(provincial)

print("前処理完了")
print(f"追加されたカラム: YEAR_START, YEAR_END, YEAR_MIDPOINT")

## 2. 探索的データ分析（EDA）

In [ ]:
# 基本統計量
print("=== デナリウス貨の成分データ統計 ===")
loader.get_summary_stats(denarii)

In [ ]:
# 皇帝別の銀含有率統計
print("=== 皇帝別の銀含有率 ===")
emperor_stats = loader.get_emperor_stats(denarii)
emperor_stats

In [ ]:
# 皇帝の出現頻度
print("=== 皇帝別のデータ件数 ===")
denarii['EMPEROR'].value_counts()

In [ ]:
# 鋳造地の分布
if 'MINT' in denarii.columns:
    print("=== 鋳造地別のデータ件数 ===")
    print(denarii['MINT'].value_counts().head(10))

## 3. 銀含有率の時系列分析

In [ ]:
# 時系列データの生成
ts = loader.get_time_series(denarii)
print("=== 時系列データ ===")
ts

In [ ]:
# 時系列トレンド分析（線形回帰）
slope, intercept, r_value, p_value, std_err = stats.linregress(
    ts['year'].dropna(), ts['silver_mean'].dropna()
)

print("=== 時系列トレンド分析 ===")
print(f"傾き: {slope:.4f} (%/年)")
print(f"切片: {intercept:.2f}")
print(f"決定係数 R²: {r_value**2:.4f}")
print(f"p値: {p_value:.4e}")
print(f"\n解釈: 銀含有率は年間約{abs(slope):.3f}%ずつ{'減少' if slope < 0 else '増加'}")

## 4. 統計検定

In [ ]:
# 皇帝間のANOVA検定
groups = [group['SILVER'].dropna().values 
          for name, group in denarii.groupby('EMPEROR')]

f_stat, p_value = stats.f_oneway(*groups)

print("=== 一元配置分散分析（ANOVA）===")
print(f"F統計量: {f_stat:.2f}")
print(f"p値: {p_value:.4e}")
print(f"\n結論: 皇帝間で銀含有率に{'有意差あり' if p_value < 0.05 else '有意差なし'} (α=0.05)")

In [ ]:
# ネロ前後での比較（t検定）
# ネロの在位: AD54-68
before_nero = denarii[denarii['YEAR_MIDPOINT'] < 54]['SILVER'].dropna()
after_nero = denarii[denarii['YEAR_MIDPOINT'] >= 68]['SILVER'].dropna()

t_stat, p_value = stats.ttest_ind(before_nero, after_nero)

print("=== ネロ前後の銀含有率比較（t検定）===")
print(f"ネロ以前の平均: {before_nero.mean():.2f}%")
print(f"ネロ以降の平均: {after_nero.mean():.2f}%")
print(f"t統計量: {t_stat:.2f}")
print(f"p値: {p_value:.4e}")
print(f"\n結論: ネロの貨幣改変による{'有意な低下' if p_value < 0.05 else '有意差なし'}")

## 5. 可視化

In [ ]:
# 可視化クラスの初期化
viz = SilverContentVisualizer(figsize=(12, 6))

In [ ]:
# 5.1 銀含有率の時系列グラフ
fig, ax = viz.plot_emperor_timeline(ts)
plt.show()

In [ ]:
# 5.2 皇帝別ボックスプロット
fig, ax = viz.plot_emperor_boxplot(denarii)
plt.show()

In [ ]:
# 5.3 鋳造地別散布図
fig, ax = viz.plot_scatter_by_mint(denarii)
plt.show()

In [ ]:
# 5.4 成分相関ヒートマップ
fig, ax = viz.plot_correlation_heatmap(denarii)
plt.show()

In [ ]:
# 5.5 銀含有率の分布
fig, ax = viz.plot_silver_distribution(denarii)
plt.show()

## 6. 図の一括保存

In [ ]:
# 全ての図を outputs/figures/ に保存
create_all_figures(denarii, ts)
print(f"\n保存先: {FIGURES_DIR}")

## 7. 3世紀データの分析

3世紀の危機（AD235-284年）における銀含有率の急落を分析

In [ ]:
# 3世紀データの読み込み
third_century = loader.load_third_century()
print(f"3世紀データ件数: {len(third_century)}")
print("\nデータサンプル:")
third_century.head(10)

In [ ]:
# 完全な時系列データ（BC44〜AD295）
full_ts = loader.get_full_time_series()
print(f"完全な時系列データ: {len(full_ts)} 件")
print(f"期間: AD{full_ts['year'].min()} 〜 AD{full_ts['year'].max()}")
full_ts

In [ ]:
# BC44〜AD295の銀含有率推移グラフ
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(full_ts['year'], full_ts['silver'], 'o-', markersize=6, linewidth=2, color='#2E86AB')

# 重要イベントをマーク
events = {64: 'ネロの貨幣改変', 215: 'アントニニアヌス導入', 235: '3世紀の危機開始', 274: 'アウレリアヌス改革'}
for year, event in events.items():
    ax.axvline(x=year, color='red', linestyle='--', alpha=0.5)
    ax.text(year+2, 95, event, rotation=90, va='top', fontsize=9, color='red')

ax.set_xlabel('年 (AD)', fontsize=12)
ax.set_ylabel('銀含有率 (%)', fontsize=12)
ax.set_title('ローマ銀貨の銀含有率推移（BC27〜AD295）', fontsize=14)
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.3)

# 3世紀の危機期間をハイライト
ax.axvspan(235, 284, alpha=0.2, color='orange', label='3世紀の危機')
ax.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / "full_silver_timeline.png", dpi=300)
plt.show()

## 8. ロトカ・ヴォルテラモデルによるシミュレーション

Roman & Palmer (2019) を参考にした数理モデル

In [ ]:
# ロトカ・ヴォルテラモデルのインポート
from src.lotka_volterra import RomanEmpireModel, plot_simulation_results

# モデル初期化
model = RomanEmpireModel()
print("モデルパラメータ:")
for k, v in model.params.items():
    print(f"  {k}: {v}")

In [ ]:
# シミュレーション実行（歴史的銀含有率データを使用）
results = model.run_simulation(
    start_year=-27,
    end_year=300,
    silver_data=full_ts
)

print("シミュレーション結果:")
print(f"期間: BC{abs(results['year'].min())} 〜 AD{results['year'].max()}")
results[results['year'].isin([-27, 0, 100, 200, 235, 268, 284, 300])]

In [ ]:
# シミュレーション結果のプロット
fig, axes = plot_simulation_results(results, save_path=FIGURES_DIR / "lotka_volterra_simulation.png")
plt.show()

## 9. まとめと考察

### 分析結果
1. **ネロの貨幣改変（AD64）**: 銀含有率が約99%から80%に低下
2. **3世紀の危機（AD235-284）**: 銀含有率が約40%から4%に急落
3. **ロトカ・ヴォルテラモデル**: 銀含有率の低下と軍隊・領土の変動を連動させてシミュレーション

### 次のステップ
- パラメータの最適化（歴史データへのフィッティング）
- 感度分析
- 論文用図表の最終調整